# Complete Implementation of the MLP

Define scalar values

In [5]:
import math
class Value:
    def __init__(self, data, children=(), op="", label=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(children)
        self._op = op
        self.label = label
        self._backward = lambda: None

    def __add__(self, other):
        # TODO: coerce scalar, create output, attach closure
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = backward
        return out
    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        # TODO
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = backward
        return out
    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        out = Value(self.data * -1, (self,), "-")
        def backward():
            self.grad += -1 * out.grad
        out._backward = backward
        return out
    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data - other.data, (self, other), "-")
        def backward():
            self.grad += out.grad
            other.grad += -1 * out.grad
        out._backward = backward
        return out
    def __rsub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other - self
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data / other.data, (self, other), "/")
        def backward():
            self.grad += (1 / other.data) * out.grad
            other.grad += (-self.data / (other.data ** 2)) * out.grad
        out._backward = backward
        return out
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other / self
    def __pow__(self, exponent):
        out = Value(self.data ** exponent, (self,), f"**{exponent}")
        def backward():
            self.grad += (exponent * self.data ** (exponent - 1)) * out.grad
        out._backward = backward
        return out
    def tanh(self):
        out = Value((math.exp(2 * self.data) - 1) / (math.exp(2 * self.data) + 1), (self,), "tanh")
        def backward():
            self.grad += (1 - out.data ** 2) * out.grad
        out._backward = backward
        return out
    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def backward():
            self.grad += out.data * out.grad
        out._backward = backward
        return out
    def log(self):
        out = Value(math.log(self.data), (self,), "log")
        def backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = backward
        return out

    def backward(self):
        '''
        Build topological order of the graph, then go one variable at a time and apply the chain rule to get its gradient.
        '''

        # the reason for topological order is that we make sure no variable is backpropagated before all of its children have been backpropagated
        from collections import deque
        visited = set()
        topo = deque()
        def dfs(v):
            nonlocal visited
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    dfs(child)
                topo.appendleft(v)

        dfs(self)
        self.grad = 1.0
        for v in topo:
            v._backward()


Define Classes for the MLP

In [6]:
class Module:
    def parameters(self):
        return []

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

class Neuron(Module):
    def __init__(self, n_in, *, nonlinear=True, rng=None):
        self.nonlinear = nonlinear
        self.w = [Value(rng.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlinear else act
    def parameters(self):
        return self.w + [self.b]

class Layer(Module):
    def __init__(self, n_in, n_out, **kwargs):
        self.neurons = [Neuron(n_in, **kwargs) for _ in range(n_out)]
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, n_in, widths, **kwargs):
        sz = [n_in] + widths
        self.layers = [Layer(sz[i], sz[i + 1], **kwargs) for i in range(len(widths))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

def train_step(mlp: MLP, input_data, target, learning_rate=0.01):
    # clear gradients
    mlp.zero_grad()

    # Forward pass
    output = mlp(input_data)

    # print("Output of the MLP:", output.data)

    loss = (output - target) ** 2  # Mean Squared Error loss

    loss.backward()  # Backpropagation

    for p in mlp.parameters():
        p.data -= learning_rate * p.grad  # Update parameters using gradient descent

Running the MLP (my implementation)

In [7]:
import random

# Set a target prediction
target = Value(0.8)

# Set a random seed for reproducibility
random.seed(42)

# Create an MLP with 3 inputs, two hidden layers of 4 neurons each, and 1 output neuron
mlp = MLP(n_in=3, widths=[4, 4, 1], nonlinear=True, rng=random)

# Output layer should be linear (no activation function)
mlp.layers[-1] = Layer(4, 1, nonlinear=False, rng=random)

# Example input
input_data = [Value(0.5), Value(-0.2), Value(0.1)]


# Forward pass through the MLP
output = mlp(input_data)

# Print the output value
print("Output of the initial MLP:", output.data)

# Perform training
for step in range(50):
    train_step(mlp, input_data, target, learning_rate=0.01)

# Forward pass through the MLP after training
output = mlp(input_data)

# Print the output value after training
print("Output of the MLP after training:", output.data)

Output of the initial MLP: 0.38296152849245085
Output of the MLP after training: 0.7806496334634204


Running the MLP (PyTorch implementation)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# 1. Define a PyTorch MLP class
class PyTorchMLP(nn.Module):
    def __init__(self, n_in, widths, nonlinear=True):
        super().__init__()
        self.layers = nn.ModuleList()
        sz = [n_in] + widths

        for i in range(len(widths)):
            layer_in = sz[i]
            layer_out = sz[i+1]
            is_output_layer = (i == len(widths) - 1)
            # Last layer should be linear (no activation) as per the original setup
            use_activation = nonlinear if not is_output_layer else False

            linear_layer = nn.Linear(layer_in, layer_out)
            self.layers.append(linear_layer)
            if use_activation:
                self.layers.append(nn.Tanh())

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# 2. Re-initialize the Value based mlp to get its initial weights
# Set random seed again to ensure the exact same weight initialization sequence as the original.
random.seed(42)

# Re-create the Value-based MLP exactly as it was initialized in cell aokOKHJE2t-q
# This temporary MLP is used solely to extract the initial weights.
temp_value_mlp = MLP(n_in=3, widths=[4, 4, 1], nonlinear=True, rng=random)
temp_value_mlp.layers[-1] = Layer(4, 1, nonlinear=False, rng=random) # Ensure output layer is linear

# 3. Extract these initial weights and biases from the Value based mlp.
# 4. Initialize the PyTorch MLP and set its weights and biases.

# Create PyTorch MLP instance
pytorch_mlp = PyTorchMLP(n_in=3, widths=[4, 4, 1], nonlinear=True)

# Get the linear layers from the PyTorch MLP to set weights
pytorch_linear_layers = [layer for layer in pytorch_mlp.layers if isinstance(layer, nn.Linear)]

# Manually transfer weights and biases from Value-based MLP to PyTorch MLP
with torch.no_grad(): # Disable gradient calculations for weight initialization
    # Layer 1: n_in=3, n_out=4
    for i, neuron in enumerate(temp_value_mlp.layers[0].neurons):
        pytorch_linear_layers[0].weight[i] = torch.tensor([w.data for w in neuron.w], dtype=torch.float32)
        pytorch_linear_layers[0].bias[i] = torch.tensor(neuron.b.data, dtype=torch.float32)

    # Layer 2: n_in=4, n_out=4
    for i, neuron in enumerate(temp_value_mlp.layers[1].neurons):
        pytorch_linear_layers[1].weight[i] = torch.tensor([w.data for w in neuron.w], dtype=torch.float32)
        pytorch_linear_layers[1].bias[i] = torch.tensor(neuron.b.data, dtype=torch.float32)

    # Layer 3 (Output Layer): n_in=4, n_out=1
    # There's only one neuron in the last layer
    neuron = temp_value_mlp.layers[2].neurons[0]
    pytorch_linear_layers[2].weight[0] = torch.tensor([w.data for w in neuron.w], dtype=torch.float64)
    pytorch_linear_layers[2].bias[0] = torch.tensor(neuron.b.data, dtype=torch.float64)

# 5. Convert input_data and target to PyTorch tensors.
# Original input_data: [Value(0.5), Value(-0.2), Value(0.1)]
# Original target: Value(0.8)
pytorch_input_data = torch.tensor([0.5, -0.2, 0.1], dtype=torch.float32).unsqueeze(0) # Add batch dimension
pytorch_target = torch.tensor([0.8], dtype=torch.float32).unsqueeze(0) # Add batch dimension

# 6. Perform the same number of training steps.

# Loss function and optimizer (using MSE Loss and SGD as in the original setup)
criterion = nn.MSELoss()
optimizer = optim.SGD(pytorch_mlp.parameters(), lr=0.01) # learning_rate=0.01

# Initial forward pass through PyTorch MLP
initial_pytorch_output = pytorch_mlp(pytorch_input_data)
print("Output of the initial PyTorch MLP:", initial_pytorch_output.item())

num_training_steps = 50
for step in range(num_training_steps):
    optimizer.zero_grad() # Clear gradients from previous step
    output = pytorch_mlp(pytorch_input_data)
    loss = criterion(output, pytorch_target)
    loss.backward() # Compute gradients
    optimizer.step() # Update model parameters

# Final forward pass through PyTorch MLP after training
final_pytorch_output = pytorch_mlp(pytorch_input_data)
print("Output of the PyTorch MLP after training:", final_pytorch_output.item())


Output of the initial PyTorch MLP: 0.38296157121658325
Output of the PyTorch MLP after training: 0.780649721622467
